# LeetCode 2310 — Sum of Numbers With Units Digit K bằng Python

Link đề: https://leetcode.com/problems/sum-of-numbers-with-units-digit-k/ · Độ khó: **Medium** · Chủ đề: Math, Dynamic Programming, Greedy, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def minimumNumbers(self, num: int, k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Viết mỗi số có chữ số hàng đơn vị `k` dưới dạng `10·x + k` và rút ra: tổng của `n` số như vậy là `n·k + 10·(Σx)`.
2. Điều kiện **cần và đủ** để có `n` số: `num − n·k ≥ 0` và chia hết cho 10 (cộng một điểm tinh tế khi `k = 0`).
3. Dùng **tính tuần hoàn modulo 10**: `n·k mod 10` lặp chu kỳ ≤ 10 ⇒ chỉ cần thử `n = 1 … 10`.
4. Đối chiếu với DP "đổi tiền" (unbounded knapsack) trên tập đồng xu `{k, 10 + k, 20 + k, …}` để kiểm chứng.
5. Xử lý đủ các trường hợp biên: `num = 0`, `k = 0`, `n·k > num`.

## 2. Đề bài & ví dụ

Cho `num` và `k`. Xét một tập (đa tập) các số nguyên **dương** sao cho: mỗi số có chữ số hàng đơn vị là `k`, và tổng các số bằng `num`. Trả về **kích thước nhỏ nhất** của tập như vậy, hoặc `−1` nếu không tồn tại. Tập có thể chứa một số nhiều lần; tổng của tập rỗng là 0.

**Ví dụ 1:** `num = 58`, `k = 9` → `2` (`[9, 49]` hoặc `[19, 39]`).

**Ví dụ 2:** `num = 37`, `k = 2` → `−1`.

**Ví dụ 3:** `num = 0`, `k = 7` → `0` (tập rỗng).

**Ràng buộc:** `0 ≤ num ≤ 3000`, `0 ≤ k ≤ 9`.

In [1]:
from functools import lru_cache
from math import inf, gcd
import random, time

examples = [(58, 9, 2), (37, 2, -1), (0, 7, 0)]
for num, k, exp in examples:
    print(f"num={num} k={k} -> kỳ vọng {exp}")

num=58 k=9 -> kỳ vọng 2
num=37 k=2 -> kỳ vọng -1
num=0 k=7 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Số có hàng đơn vị `k`

`x mod 10 = k` ⇔ `x = 10·q + k` với `q ≥ 0`. Với `k ≥ 1`, số nhỏ nhất là `k` (q = 0). Với `k = 0`, vì phải **dương**, số nhỏ nhất là `10` (q ≥ 1).

In [2]:
for k in [9, 2, 0]:
    print(k, "->", [x for x in range(1, 60) if x % 10 == k][:6])

9 -> [9, 19, 29, 39, 49, 59]
2 -> [2, 12, 22, 32, 42, 52]
0 -> [10, 20, 30, 40, 50]


### 3.2 Tổng của `n` số như vậy

`Σ (10·qᵢ + k) = n·k + 10·Σqᵢ` ⇒ `tổng ≡ n·k (mod 10)` và `tổng ≥ n·k`.

### 3.3 Chu kỳ của `n·k mod 10`

Dãy `k, 2k, 3k, … (mod 10)` có chu kỳ chia hết 10 (vì `10·k ≡ 0`). Vì vậy nếu có `n` thỏa `n·k ≡ num (mod 10)` thì có một `n` như thế trong `1 … 10`.

In [3]:
for k in range(10):
    print(k, [(n * k) % 10 for n in range(1, 11)])

0 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
1 [1, 2, 3, 4, 5, 6, 7, 8, 9, 0]
2 [2, 4, 6, 8, 0, 2, 4, 6, 8, 0]
3 [3, 6, 9, 2, 5, 8, 1, 4, 7, 0]
4 [4, 8, 2, 6, 0, 4, 8, 2, 6, 0]
5 [5, 0, 5, 0, 5, 0, 5, 0, 5, 0]
6 [6, 2, 8, 4, 0, 6, 2, 8, 4, 0]
7 [7, 4, 1, 8, 5, 2, 9, 6, 3, 0]
8 [8, 6, 4, 2, 0, 8, 6, 4, 2, 0]
9 [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]


Đọc bảng: với `k = 2` các chữ số cuối đạt được chỉ là số chẵn ⇒ `num = 37` (lẻ) không thể. Với `k = 5` chỉ đạt `0, 5`. Với `k ∈ {1, 3, 7, 9}` đạt mọi chữ số (vì `gcd(k, 10) = 1`).

### 3.4 Unbounded knapsack ("đổi tiền" với số đồng xu ít nhất)

`dp[v] = 1 + min dp[v − c]` trên các đồng xu `c ≤ v`. Dùng làm bản kiểm chứng độc lập.

## 4. Ý tưởng nền tảng / trực giác

**Bổ đề:** với `num > 0` và `k ≥ 1`, tồn tại tập kích thước `n ≥ 1` ⇔ `n·k ≤ num` **và** `(num − n·k) mod 10 = 0`.

- (⇒) từ mục 3.2.
- (⇐) lấy `n − 1` số bằng `k` và một số bằng `num − (n−1)·k`. Số cuối `= k + (num − n·k)` có hàng đơn vị `k` (vì `num − n·k` chia hết cho 10) và `≥ k ≥ 1` ⇒ dương.

Với `k = 0`: các số là bội dương của 10; `num > 0` cần `num mod 10 = 0`, và khi đó **một** số (`num` chính nó) là đủ. Điều kiện bổ đề với `n = 1` cho đúng kết quả này.

Vậy đáp án = `n` **nhỏ nhất** thỏa điều kiện; theo mục 3.3 chỉ cần thử `n = 1 … 10`.

Tính tay ví dụ 1 (`num = 58`, `k = 9`):

| n | n·k | num − n·k | chia hết 10? |
|---|---|---|---|
| 1 | 9 | 49 | ✘ |
| 2 | 18 | 40 | ✔ |

⇒ `2`, tập `[9, 49]` ✔. Ví dụ 2 (`37`, `2`): `n·2` luôn chẵn, `37 − chẵn` luôn lẻ ⇒ `−1`.

In [4]:
def witness(num, k, n):
    return [k] * (n - 1) + [num - (n - 1) * k]

for num, k, exp in examples:
    if exp > 0:
        w = witness(num, k, exp); print(num, k, "->", w, "tổng", sum(w), "hàng đơn vị", [x % 10 for x in w])

58 9 -> [9, 49] tổng 58 hàng đơn vị [9, 9]


### 4.1 Nhìn theo "đuôi" và "thân"

Hình dung `num` gồm hai phần: phần **đuôi** `n·k` (bắt buộc, vì mỗi số góp ít nhất `k` — với `k ≥ 1`) và phần **thân** `num − n·k` phải chia thành các khối 10 để "dán" vào bất kỳ số nào. Ta không quan tâm khối 10 dán vào đâu — chỉ cần thân không âm và chia hết cho 10. Vì vậy câu hỏi chỉ còn về **số lượng** `n`, không về giá trị từng số.

In [5]:
num, k = 58, 9
for n in range(1, 4):
    body = num - n * k
    print(f"n={n}: đuôi {n}*{k} = {n*k:<3} thân = {body:<3} -> {'chia hết 10' if body >= 0 and body % 10 == 0 else 'không hợp lệ'}")

n=1: đuôi 1*9 = 9   thân = 49  -> không hợp lệ
n=2: đuôi 2*9 = 18  thân = 40  -> chia hết 10
n=3: đuôi 3*9 = 27  thân = 31  -> không hợp lệ


## 5. Cách 1 — Brute Force / baseline

### 5.1 DP đổi tiền trên các đồng xu `{k, 10+k, 20+k, …} ∩ [1, num]` — O(num² / 10)

In [6]:
def brute(num, k):
    coins = [c for c in range(1, num + 1) if c % 10 == k]
    dp = [0] + [inf] * num
    for v in range(1, num + 1):
        for c in coins:
            if c > v: break
            if dp[v - c] + 1 < dp[v]: dp[v] = dp[v - c] + 1
    return -1 if dp[num] == inf else dp[num]

for num, k, exp in examples:
    print(brute(num, k), "OK" if brute(num, k) == exp else "SAI")

2 OK
-1 OK
0 OK


Đúng theo định nghĩa (đa tập ⇒ đồng xu dùng lại được). `num = 3000` ⇒ ~`3000 · 300 = 9·10⁵` phép — chạy được nhưng chậm, và không cho thấy cấu trúc của bài.

### 5.2 Baseline 2 — Thử mọi `n` từ 1 đến `num` (cách doocs 1) — O(num)

In [7]:
def enumerate_all(num, k):
    if num == 0: return 0
    for n in range(1, num + 1):
        t = num - k * n
        if t >= 0 and t % 10 == 0: return n
    return -1

for num, k, exp in examples:
    print(enumerate_all(num, k), "OK" if enumerate_all(num, k) == exp else "SAI")

2 OK
-1 OK
0 OK


**Nút thắt:** khi không có nghiệm, vòng lặp chạy hết `num` lần dù đã biết sau 10 lần là vô vọng (chu kỳ mod 10).

## 6. Ý tưởng thuật toán / bước đột phá

```
if num == 0: return 0
for n in 1..10:
    if n*k <= num and (num - n*k) % 10 == 0: return n
return -1
```

**Vì sao 10 là đủ?** Nếu `n > 10` thỏa thì `n' = n − 10` cũng thỏa: `n'·k ≡ n·k (mod 10)` và `n'·k < n·k ≤ num`. Lặp lại tới khi `n ∈ [1, 10]`. (Với `n' = 0` thì `num ≡ 0` và `num > 0` — khi đó `n = 10` vẫn nằm trong `[1, 10]`, nhưng thực ra `n` nhỏ hơn cũng có thể đúng; vòng lặp sẽ tìm cái nhỏ nhất.)

**Độ phức tạp:** O(1).

### 6.1 Công thức đóng bằng nghịch đảo modulo (khi `gcd(k, 10) = 1`)

Với `k ∈ {1, 3, 7, 9}`, `k` khả nghịch modulo 10: `n ≡ num · k⁻¹ (mod 10)`. Lấy `n₀ ∈ [1, 10]` thuộc lớp đó (thay 0 bằng 10). Đáp án `n₀` nếu `n₀·k ≤ num`, ngược lại −1. Với `k ∈ {0, 2, 4, 5, 6, 8}` (không khả nghịch) cần thêm điều kiện `gcd(k, 10) | num` rồi giải đồng dư rút gọn — vòng thử 10 giá trị gọn hơn nhiều, nên công thức này chủ yếu để hiểu cấu trúc. (Cell dưới được đối chiếu với lời giải chính ở mục 12.)

In [8]:
def closed_form_coprime(num, k):
    assert k in (1, 3, 7, 9)
    if num == 0: return 0
    inv = pow(k, -1, 10)                      # nghịch đảo modulo 10
    n0 = (num * inv) % 10 or 10
    return n0 if n0 * k <= num else -1

print({k: pow(k, -1, 10) for k in (1, 3, 7, 9)}, closed_form_coprime(58, 9), closed_form_coprime(10, 1), closed_form_coprime(4, 9))

{1: 1, 3: 7, 7: 3, 9: 9} 2 10 -1


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Thử `n = 1 … 10` (cách doocs 2)

In [9]:
def enumerate_ten(num, k):
    if num == 0: return 0
    for n in range(1, 11):
        if (k * n) % 10 == num % 10 and k * n <= num: return n
    return -1

for num, k, exp in examples:
    print(enumerate_ten(num, k), "OK" if enumerate_ten(num, k) == exp else "SAI")

2 OK
-1 OK
0 OK


### 7.2 Cách 3 — Đệ quy có nhớ trừ dần các số có hàng đơn vị `k` (cách doocs 3)

In [10]:
def memo(num, k):
    if num == 0: return 0
    if k == 0: return -1 if num % 10 else 1
    @lru_cache(maxsize=None)
    def dfs(v):
        if v == 0: return 0
        if v < 10 and v % k: return inf
        t = inf; i = 0
        while (x := i * 10 + k) <= v:
            t = min(t, dfs(v - x)); i += 1
        return t + 1
    ans = dfs(num); dfs.cache_clear()
    return -1 if ans >= inf else ans

for num, k, exp in examples:
    print(memo(num, k), "OK" if memo(num, k) == exp else "SAI")

2 OK
-1 OK
0 OK


**Cảnh báo độ sâu đệ quy:** với `k = 1`, chuỗi gọi `dfs(num) → dfs(num − 1) → …` sâu tới `num` tầng; `num = 3000` vượt giới hạn mặc định 1000 của Python ⇒ `RecursionError`. Trên LeetCode bản này vẫn qua vì môi trường đặt giới hạn cao hơn, nhưng trong notebook ta chỉ chạy nó với `num ≤ 600` (cell dưới minh họa lỗi một cách an toàn).

In [11]:
try:
    memo(3000, 1)
    print("không lỗi (giới hạn đệ quy của môi trường đủ lớn)")
except RecursionError as e:
    print("RecursionError:", str(e)[:60])
print("memo(600, 1) =", memo(600, 1), "| memo(58, 9) =", memo(58, 9))

RecursionError: maximum recursion depth exceeded
memo(600, 1) = 10 | memo(58, 9) = 2


Chậm hơn nhiều (O(num²/10)) nhưng **tổng quát hóa** được khi tập đồng xu không có cấu trúc đẹp (mục 15.3). Lưu ý dòng `if v < 10 and v % k` là một cắt tỉa: `v < 10` thì chỉ dùng được đồng xu `k`, nên `v` phải là bội của `k`.

### 7.3 Cách 4 — Bảng tra theo `(k, num mod 10)`

Với mỗi `k` và chữ số cuối `d`, `n₀(k, d)` = `n ∈ [1, 10]` nhỏ nhất có `n·k ≡ d`. Đáp án = `n₀` nếu `n₀·k ≤ num`, ngược lại `−1`. Vì sao "ngược lại −1"? Mọi `n` khác thỏa đồng dư đều `≥ n₀` (là `n₀ + 10t`) nên `n·k ≥ n₀·k > num`.

In [12]:
TABLE = [[next((n for n in range(1, 11) if (n * k) % 10 == d), None) for d in range(10)] for k in range(10)]

def table_lookup(num, k):
    if num == 0: return 0
    n0 = TABLE[k][num % 10]
    return n0 if n0 is not None and n0 * k <= num else -1

for k in range(10): print(k, TABLE[k])
for num, k, exp in examples:
    print(table_lookup(num, k), "OK" if table_lookup(num, k) == exp else "SAI")

0 [1, None, None, None, None, None, None, None, None, None]
1 [10, 1, 2, 3, 4, 5, 6, 7, 8, 9]
2 [5, None, 1, None, 2, None, 3, None, 4, None]
3 [10, 7, 4, 1, 8, 5, 2, 9, 6, 3]
4 [5, None, 3, None, 1, None, 4, None, 2, None]
5 [2, None, None, None, None, 1, None, None, None, None]
6 [5, None, 2, None, 4, None, 1, None, 3, None]
7 [10, 3, 6, 9, 2, 5, 8, 1, 4, 7]
8 [5, None, 4, None, 3, None, 2, None, 1, None]
9 [10, 9, 8, 7, 6, 5, 4, 3, 2, 1]
2 OK
-1 OK
0 OK


Lưu ý với `k = 0`: `TABLE[0][0] = 1` (vì `1·0 ≡ 0`) và `TABLE[0][d ≠ 0] = None` — đúng như phân tích: bội của 10 cần đúng một số.

## 8. TRACE từng bước

### 8.1 Trace cách 2 trên vài bộ

In [13]:
def trace(num, k):
    print(f"num={num}, k={k}:")
    if num == 0: print("  num = 0 -> tập rỗng, đáp án 0"); return 0
    for n in range(1, 11):
        ok_mod = (k * n) % 10 == num % 10; ok_size = k * n <= num
        print(f"  n={n:<2} n*k={n * k:<3} (mod 10 = {(n * k) % 10}) vs num mod 10 = {num % 10}: đồng dư {ok_mod!s:<5} n*k<=num {ok_size}")
        if ok_mod and ok_size: print("  -> đáp án", n, "tập", witness(num, k, n)); return n
    print("  -> -1"); return -1

for num, k in [(58, 9), (37, 2), (10, 1), (4, 9)]:
    trace(num, k)

num=58, k=9:
  n=1  n*k=9   (mod 10 = 9) vs num mod 10 = 8: đồng dư False n*k<=num True
  n=2  n*k=18  (mod 10 = 8) vs num mod 10 = 8: đồng dư True  n*k<=num True
  -> đáp án 2 tập [9, 49]
num=37, k=2:
  n=1  n*k=2   (mod 10 = 2) vs num mod 10 = 7: đồng dư False n*k<=num True
  n=2  n*k=4   (mod 10 = 4) vs num mod 10 = 7: đồng dư False n*k<=num True
  n=3  n*k=6   (mod 10 = 6) vs num mod 10 = 7: đồng dư False n*k<=num True
  n=4  n*k=8   (mod 10 = 8) vs num mod 10 = 7: đồng dư False n*k<=num True
  n=5  n*k=10  (mod 10 = 0) vs num mod 10 = 7: đồng dư False n*k<=num True
  n=6  n*k=12  (mod 10 = 2) vs num mod 10 = 7: đồng dư False n*k<=num True
  n=7  n*k=14  (mod 10 = 4) vs num mod 10 = 7: đồng dư False n*k<=num True
  n=8  n*k=16  (mod 10 = 6) vs num mod 10 = 7: đồng dư False n*k<=num True
  n=9  n*k=18  (mod 10 = 8) vs num mod 10 = 7: đồng dư False n*k<=num True
  n=10 n*k=20  (mod 10 = 0) vs num mod 10 = 7: đồng dư False n*k<=num True
  -> -1
num=10, k=1:
  n=1  n*k=1   (mod 10 = 1)

`num = 10, k = 1` cần **đúng 10 số** (mười số `1`) — trường hợp này cho thấy vòng lặp phải chạy tới 10, không phải 9. `num = 4, k = 9`: đồng dư đạt ở `n = 6` (`54 ≡ 4`) nhưng `54 > 4` ⇒ `−1`.

### 8.2 So sánh với nghiệm DP trên một dải `num`

In [14]:
k = 3
print("num:", list(range(0, 31)))
print("DP :", [brute(v, k) for v in range(0, 31)])
print("O(1):", [enumerate_ten(v, k) for v in range(0, 31)])

num: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
DP : [0, -1, -1, 1, -1, -1, 2, -1, -1, 3, -1, -1, 4, 1, -1, 5, 2, -1, 6, 3, -1, 7, 4, 1, 8, 5, 2, 9, 6, 3, 10]
O(1): [0, -1, -1, 1, -1, -1, 2, -1, -1, 3, -1, -1, 4, 1, -1, 5, 2, -1, 6, 3, -1, 7, 4, 1, 8, 5, 2, 9, 6, 3, 10]


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def minimumNumbers(self, num: int, k: int) -> int:
        if num == 0:
            return 0                                   # tập rỗng
        for n in range(1, 11):                         # n*k mod 10 tuần hoàn chu kỳ <= 10
            if n * k <= num and (num - n * k) % 10 == 0:
                return n                               # n-1 số k và một số num-(n-1)k
        return -1

In [16]:
sol = Solution()
for num, k, exp in examples:
    got = sol.minimumNumbers(num, k)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

2 | kỳ vọng 2 | OK
-1 | kỳ vọng -1 | OK
0 | kỳ vọng 0 | OK


## 10. Vì sao thuật toán đúng

**Bổ đề A (cần).** Nếu có đa tập `n ≥ 1` số dương có hàng đơn vị `k`, tổng `num`, thì `n·k ≤ num` và `10 | num − n·k`. *Chứng minh:* mỗi số `= 10qᵢ + k` với `qᵢ ≥ 0` ⇒ tổng `= n·k + 10Σqᵢ`.

**Bổ đề B (đủ).** Nếu `num > 0`, `n ≥ 1`, `n·k ≤ num`, `10 | num − n·k`, thì có đa tập kích thước `n`.
- `k ≥ 1`: `n − 1` số `k` và số `num − (n−1)k = k + (num − nk)` — hàng đơn vị `k`, giá trị `≥ k ≥ 1`.
- `k = 0`: `num` là bội dương của 10; với `n = 1` chọn `{num}`. (Với `n ≥ 2` cũng được nếu `num ≥ 10n`, nhưng ta chỉ cần `n` nhỏ nhất, là 1.) Lưu ý thuật toán với `k = 0` trả về `n = 1` ngay khi `num mod 10 = 0` — đúng.

**Bổ đề C (đủ thử tới 10).** Nếu `n > 10` thỏa điều kiện của B thì `n − 10` cũng thỏa (đồng dư giữ nguyên, `(n−10)k ≤ nk ≤ num`) và `n − 10 ≥ 1`. ⇒ `n` nhỏ nhất (nếu tồn tại) nằm trong `[1, 10]`.

**Bổ sung — chu kỳ thật.** Dãy `n·k mod 10` có chu kỳ đúng bằng `p = 10 / gcd(k, 10)`: `p = 10` với `k ∈ {1, 3, 7, 9}`, `p = 5` với `k ∈ {2, 4, 6, 8}`, `p = 2` với `k = 5`, `p = 1` với `k = 0`. Bổ đề C dùng bước 10 (bội của mọi `p`) nên luôn đúng; nhưng nếu muốn liệt kê **mọi** kích thước khả thi (bài tập 5) thì phải nhảy theo `p`, không phải 10. Ngoài ra, điều kiện đồng dư `n·k ≡ num (mod 10)` có nghiệm ⇔ `gcd(k, 10) | num` — đó là lý do `k` chẵn không bao giờ cho tổng lẻ, và `k = 5` chỉ cho tổng tận cùng 0 hoặc 5.

**Kết luận.** `num = 0` ⇒ tập rỗng, 0. `num > 0` ⇒ đáp án là `n ∈ [1, 10]` nhỏ nhất thỏa B (tối thiểu theo A), hoặc `−1` nếu không có. ∎

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. DP đổi tiền | O(num² / 10) | O(num) |
| 1b. Thử `n = 1 … num` | O(num) | O(1) |
| 2. Thử `n = 1 … 10` | O(1) | O(1) |
| 3. Đệ quy có nhớ | O(num² / 10) | O(num) |
| 4. Bảng tra | O(1) (bảng 10 × 10) | O(1) |

In [17]:
res = {}
for name, f in [("enumerate_all", enumerate_all), ("enumerate_ten", enumerate_ten), ("table_lookup", table_lookup),
                ("Solution", Solution().minimumNumbers)]:
    t0 = time.perf_counter(); out = [f(v, k) for v in range(3001) for k in range(10)]; dt = time.perf_counter() - t0
    res[name] = out; print(f"{name:<14} toàn bộ 3001 x 10 bộ: {dt*1000:7.1f} ms")
assert len({tuple(v) for v in res.values()}) == 1
t0 = time.perf_counter(); brute(3000, 7); print(f"brute (num=3000): {(time.perf_counter()-t0)*1000:.0f} ms cho MỘT bộ")

enumerate_all  toàn bộ 3001 x 10 bộ:   745.1 ms
enumerate_ten  toàn bộ 3001 x 10 bộ:     9.0 ms
table_lookup   toàn bộ 3001 x 10 bộ:     2.4 ms
Solution       toàn bộ 3001 x 10 bộ:    12.5 ms
brute (num=3000): 19 ms cho MỘT bộ


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = {"brute": brute, "enumerate_all": enumerate_all, "enumerate_ten": enumerate_ten, "memo": memo,
       "table_lookup": table_lookup, "Solution": Solution().minimumNumbers}
edge = examples + [
    (0, 0, 0),            # tập rỗng kể cả k = 0
    (10, 0, 1),           # {10}
    (5, 0, -1),           # bội của 10 không thể tổng 5
    (10, 1, 10),          # mười số 1
    (4, 9, -1),           # đồng dư ở n = 6 nhưng 54 > 4
    (9, 9, 1),
    (1, 1, 1),
    (2, 1, 2),
    (3000, 5, 2),         # 5 + 2995
    (3000, 2, 5),         # 5*2 = 10 ≡ 0
    (2999, 1, 9),
]
for num, k, exp in edge:
    for name, f in ALL.items():
        if name == "memo" and num > 600: continue          # độ sâu đệ quy ~ num/k (xem 7.2)
        assert f(num, k) == exp, (name, num, k, f(num, k), exp)
    print(f"num={num:<5} k={k} -> {exp}")
print("edge OK")

num=58    k=9 -> 2
num=37    k=2 -> -1
num=0     k=7 -> 0
num=0     k=0 -> 0
num=10    k=0 -> 1
num=5     k=0 -> -1
num=10    k=1 -> 10
num=4     k=9 -> -1
num=9     k=9 -> 1
num=1     k=1 -> 1
num=2     k=1 -> 2
num=3000  k=5 -> 2
num=3000  k=2 -> 5
num=2999  k=1 -> 9
edge OK


### 12.2 Đối chiếu toàn bộ với DP trên `num ≤ 400`, mọi `k`

In [19]:
for k in range(10):
    coins = [c for c in range(1, 401) if c % 10 == k]
    dp = [0] + [inf] * 400
    for v in range(1, 401):
        for c in coins:
            if c > v: break
            dp[v] = min(dp[v], dp[v - c] + 1)
    for v in range(401):
        ref = -1 if dp[v] == inf else dp[v]
        assert Solution().minimumNumbers(v, k) == ref == table_lookup(v, k), (v, k)
for k in (1, 3, 7, 9):
    for v in range(0, 3001):
        assert closed_form_coprime(v, k) == Solution().minimumNumbers(v, k)
print("đối chiếu đầy đủ num <= 400, k = 0..9 OK; công thức nghịch đảo modulo khớp với num <= 3000")

đối chiếu đầy đủ num <= 400, k = 0..9 OK; công thức nghịch đảo modulo khớp với num <= 3000


### 12.3 Tính chất: nghiệm luôn ≤ 10; tập chứng nhận hợp lệ; với `gcd(k, 10) = 1` và `num ≥ 9k` luôn có nghiệm

In [20]:
for k in range(10):
    for v in range(1, 3001):
        n = Solution().minimumNumbers(v, k)
        assert n == -1 or 1 <= n <= 10
        if n > 0:
            w = witness(v, k, n) if k else [v]
            assert sum(w) == v and all(x > 0 and x % 10 == k for x in w) and len(w) == n
        if gcd(k, 10) == 1 and v >= 10 * k:
            assert n != -1
print("tính chất OK")

tính chất OK


Vì sao `gcd(k, 10) = 1` và `num ≥ 10k` luôn có nghiệm? `n ↦ n·k mod 10` là song ánh trên `{1..10}` ⇒ luôn có `n ≤ 10` đồng dư, và `n·k ≤ 10k ≤ num`.

### 12.4 Edge cực trị: bảng đáp án cho `num = 3000` và `num = 2999` với mọi `k`

In [21]:
for num in (3000, 2999, 1):
    row = [Solution().minimumNumbers(num, k) for k in range(10)]
    assert row == [brute(num, k) for k in range(10)] if num < 3000 else True
    print(f"num={num:<5}", {k: v for k, v in enumerate(row)})

num=3000  {0: 1, 1: 10, 2: 5, 3: 10, 4: 5, 5: 2, 6: 5, 7: 10, 8: 5, 9: 10}


num=2999  {0: -1, 1: 9, 2: -1, 3: 3, 4: -1, 5: -1, 6: -1, 7: 7, 8: -1, 9: 1}
num=1     {0: -1, 1: 1, 2: -1, 3: -1, 4: -1, 5: -1, 6: -1, 7: -1, 8: -1, 9: -1}


`num = 3000`: `k = 0` cần 1 số; `k = 5` cần 2 (`5 + 2995`); `k` chẵn khác 0 cần 5 (`5k ≡ 0`); `k ∈ {1, 3, 7, 9}` cần 10. `num = 2999` (lẻ): mọi `k` chẵn và `k = 5`, `k = 0` đều −1.

### 12.5 "Tập" ở đây là **đa tập**

Đề cho phép lặp số — nếu bắt buộc các số **phân biệt** thì bài khác hẳn (ví dụ `num = 10, k = 1` sẽ không còn dùng mười số 1 được). Kiểm tra nhanh bằng vét cạn cho thấy sự khác biệt:

In [22]:
from itertools import combinations
def distinct_min(num, k):
    cands = [x for x in range(1, num + 1) if x % 10 == k]
    for r in range(0, len(cands) + 1):
        if any(sum(c) == num for c in combinations(cands, r)): return r
    return -1
for num, k in [(10, 1), (58, 9), (30, 3), (20, 5)]:
    print(f"num={num}, k={k}: đa tập {Solution().minimumNumbers(num, k)}, phân biệt {distinct_min(num, k)}")

num=10, k=1: đa tập 10, phân biệt -1
num=58, k=9: đa tập 2, phân biệt 2
num=30, k=3: đa tập 10, phân biệt -1
num=20, k=5: đa tập 2, phân biệt 2


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trường hợp `num = 0` (trả về −1 hoặc 10)

In [23]:
def wrong_zero(num, k):
    for n in range(1, 11):                                  # SAI: không xử lý num = 0
        if n * k <= num and (num - n * k) % 10 == 0: return n
    return -1

for num, k in [(0, 7), (0, 0)]:
    print("bản sai :", wrong_zero(num, k), "| bản đúng:", Solution().minimumNumbers(num, k))

bản sai : -1 | bản đúng: 0
bản sai : 1 | bản đúng: 0


**Vì sao sai:** `(0, 7)`: không `n ≥ 1` nào có `7n ≤ 0` ⇒ −1; nhưng tập rỗng hợp lệ ⇒ 0. `(0, 0)`: `n = 1` thỏa `0 ≤ 0` và `0 % 10 = 0` ⇒ bản sai trả 1, đúng phải là 0 (và `{0}` không hợp lệ vì 0 không dương).

### Lỗi 2 — Chỉ thử `n = 1 … 9`

In [24]:
def wrong_nine(num, k):
    if num == 0: return 0
    for n in range(1, 10):                                  # SAI: thiếu n = 10
        if n * k <= num and (num - n * k) % 10 == 0: return n
    return -1

for num, k in [(10, 1), (58, 9), (30, 3)]:
    print("bản sai :", wrong_nine(num, k), "| bản đúng:", Solution().minimumNumbers(num, k))

bản sai : -1 | bản đúng: 10
bản sai : 2 | bản đúng: 2
bản sai : -1 | bản đúng: 10


**Vì sao sai:** chu kỳ của `n·k mod 10` có thể đúng bằng 10 (khi `gcd(k, 10) = 1`), và `n = 10` là nghiệm duy nhất trong chu kỳ khi `num ≡ 0 (mod 10)`. Ví dụ 1 **trùng hợp** đúng.

### Lỗi 3 — Bỏ điều kiện `n·k ≤ num`

In [25]:
def wrong_no_size(num, k):
    if num == 0: return 0
    for n in range(1, 11):
        if (n * k) % 10 == num % 10: return n               # SAI: có thể n*k > num
    return -1

for num, k in [(4, 9), (58, 9), (1, 3)]:
    print("bản sai :", wrong_no_size(num, k), "| bản đúng:", Solution().minimumNumbers(num, k))

bản sai : 6 | bản đúng: -1
bản sai : 2 | bản đúng: 2
bản sai : 7 | bản đúng: -1


**Vì sao sai:** `num = 4, k = 9`: `6·9 = 54 ≡ 4` nhưng 6 số mỗi số ≥ 9 có tổng ≥ 54 > 4. **Sửa:** thêm `n·k ≤ num`.

### Lỗi 4 — Coi `0` là số hợp lệ khi `k = 0`

In [26]:
def wrong_k0_formula(num, k):
    return 0 if num % 10 == 0 and k == 0 else Solution().minimumNumbers(num, k)   # SAI: "dùng các số 0"

for num, k in [(10, 0), (20, 0)]:
    print("bản sai :", wrong_k0_formula(num, k), "| bản đúng:", Solution().minimumNumbers(num, k))

bản sai : 0 | bản đúng: 1
bản sai : 0 | bản đúng: 1


**Vì sao sai:** đề yêu cầu số **dương**, nên với `k = 0` số nhỏ nhất là 10, không phải 0; `num = 10` cần đúng một số `{10}`. Nếu cho phép số 0 thì "tập" có thể chứa vô số số 0 và câu hỏi "kích thước nhỏ nhất" mất ý nghĩa ở chiều ngược lại (kích thước lớn nhất là vô hạn).

### Lỗi 5 — Tham lam: luôn lấy số lớn nhất có hàng đơn vị `k` không vượt phần còn lại

In [27]:
def wrong_greedy(num, k):
    cnt = 0; rest = num
    while rest > 0:
        c = rest - ((rest - k) % 10) if rest >= k else -1      # số lớn nhất <= rest có đuôi k
        if c <= 0 or (k == 0 and c == 0): return -1
        rest -= c; cnt += 1
        if cnt > 100: return -1
    return cnt

for num, k in [(58, 9), (10, 1), (24, 7)]:
    print("bản sai :", wrong_greedy(num, k), "| bản đúng:", Solution().minimumNumbers(num, k))

bản sai : 2 | bản đúng: 2
bản sai : 10 | bản đúng: 10
bản sai : 2 | bản đúng: 2


**Vì sao sai (xem cell dưới):** ba bộ trên đều **trùng hợp** đúng: `58, 9` lấy 49 rồi 9; `10, 1` lấy mười số 1; `24, 7` lấy 17 rồi 7. Nhưng với `num = 15, k = 3`, tham lam lấy 13 (số lớn nhất ≤ 15 có đuôi 3), còn 2 — không biểu diễn được ⇒ −1; trong khi năm số 3 cho đúng 15. Lấy số lớn không giúp gì: điều quyết định là **số lượng** `n` phải thỏa đồng dư, còn độ lớn từng số có thể điều chỉnh tùy ý bằng các khối 10.

In [28]:
bad = [(v, k) for k in range(10) for v in range(0, 300) if wrong_greedy(v, k) != Solution().minimumNumbers(v, k)]
print("số bộ tham lam sai (num < 300):", len(bad), "| vài ví dụ:", bad[:5])
v, k = bad[0]
print(f"num={v}, k={k}: tham lam {wrong_greedy(v, k)}, đúng {Solution().minimumNumbers(v, k)}")

số bộ tham lam sai (num < 300): 794 | vài ví dụ: [(15, 3), (18, 3), (21, 3), (24, 3), (25, 3)]
num=15, k=3: tham lam -1, đúng 5


**Sửa:** dùng điều kiện đồng dư — không cần chọn giá trị cụ thể của từng số.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Các số có **chữ số cuối** cố định" ⇒ viết `10q + k` ⇒ bài chuyển về đồng dư modulo 10.
- "Số lượng ít nhất" + đồng dư ⇒ thử **một chu kỳ** (≤ modulus) giá trị của `n`.
- Luôn kiểm tra cả **chặn dưới kích thước** (`n·k ≤ num`) lẫn đồng dư.
- Biên: tập rỗng (`num = 0`), `k = 0` (số nhỏ nhất là 10), `gcd(k, 10)`.
- Có DP đổi tiền làm lưới an toàn khi không chắc công thức.
- Chu kỳ của `n·k mod m` là `m / gcd(k, m)`; điều kiện có nghiệm là `gcd(k, m) | num` — hai sự kiện số học dùng lại rất nhiều (bài về đồng hồ, xoay vòng mảng, nhảy bước cố định trên vòng tròn).
- Bài cùng họ: 322 Coin Change, 279 Perfect Squares, 1015 Smallest Integer Divisible by K, 2457 Minimum Addition to Make Integer Beautiful, 1414 Find the Minimum Number of Fibonacci Numbers Whose Sum Is K.

## 15. Biến thể của bài

### 15.1 Biến thể A — Kích thước **lớn nhất** của tập

In [29]:
def maximum_numbers(num, k):
    if num == 0: return 0
    if k == 0: return num // 10 if num % 10 == 0 else -1          # mọi số là 10
    best = -1
    for n in range(1, num // k + 1):
        if (num - n * k) % 10 == 0: best = n
    return best

def maximum_brute(num, k):
    coins = [c for c in range(1, num + 1) if c % 10 == k]
    dp = [0] + [-inf] * num
    for v in range(1, num + 1):
        for c in coins:
            if c > v: break
            dp[v] = max(dp[v], dp[v - c] + 1)
    return -1 if dp[num] == -inf else dp[num]

for k in range(10):
    for v in range(0, 200):
        assert maximum_numbers(v, k) == maximum_brute(v, k), (v, k)
print(maximum_numbers(58, 9), maximum_numbers(30, 3), maximum_numbers(100, 0))

2 10 10


### 15.2 Biến thể B — Cơ số `b` bất kỳ (chữ số cuối trong hệ cơ số `b`)

Thay 10 bằng `b`: thử `n = 1 … b`.

In [30]:
def minimum_numbers_base(num, k, b):
    if num == 0: return 0
    for n in range(1, b + 1):
        if n * k <= num and (num - n * k) % b == 0 and (k > 0 or num > 0):
            return n
    return -1

def brute_base(num, k, b):
    coins = [c for c in range(1, num + 1) if c % b == k]
    dp = [0] + [inf] * num
    for v in range(1, num + 1):
        for c in coins:
            if c > v: break
            dp[v] = min(dp[v], dp[v - c] + 1)
    return -1 if dp[num] == inf else dp[num]

rng = random.Random(152)
for _ in range(400):
    b = rng.randint(2, 16); k = rng.randint(0, b - 1); v = rng.randint(0, 150)
    assert minimum_numbers_base(v, k, b) == brute_base(v, k, b), (v, k, b)
print(minimum_numbers_base(58, 9, 10), minimum_numbers_base(17, 1, 2), minimum_numbers_base(16, 0, 8))

2 1 1


### 15.3 Biến thể C — Cho phép **nhiều** chữ số cuối (tập `D`), tìm số lượng nhỏ nhất

Đồng dư trở thành: tổng các chữ số cuối đã chọn ≡ `num (mod 10)` và tổng ≤ `num`. BFS/DP trên trạng thái `(số lượng, phần dư)` với chi phí nhỏ nhất của tổng chữ số cuối.

In [31]:
def min_numbers_digits(num, D):
    if num == 0: return 0
    D = [d for d in D]
    # best[n][r] = tổng nhỏ nhất của n chữ số cuối (mỗi số có giá trị tối thiểu d, hoặc 10 nếu d = 0) có tổng mod 10 = r
    INF = inf; mins = {d: (d if d else 10) for d in D}
    best = [0 if r == 0 else INF for r in range(10)]
    for n in range(1, num + 1):
        new = [INF] * 10
        for r in range(10):
            if best[r] == INF: continue
            for d in D:
                nr = (r + d) % 10; new[nr] = min(new[nr], best[r] + mins[d])
        best = new
        if best[num % 10] <= num: return n
        if min(best) > num: return -1
    return -1

def brute_digits(num, D):
    coins = [c for c in range(1, num + 1) if c % 10 in D]
    dp = [0] + [inf] * num
    for v in range(1, num + 1):
        for c in coins:
            dp[v] = min(dp[v], dp[v - c] + 1) if c <= v else dp[v]
    return -1 if dp[num] == inf else dp[num]

rng = random.Random(153)
for _ in range(300):
    D = rng.sample(range(10), rng.randint(1, 3)); v = rng.randint(0, 120)
    assert min_numbers_digits(v, D) == brute_digits(v, D), (v, D)
print(min_numbers_digits(37, [2, 5]), min_numbers_digits(37, [2]), min_numbers_digits(58, [9]))

2 -1 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DP đổi tiền | định nghĩa | O(num²/10) | O(num) | kiểm chứng, biến thể không có cấu trúc |
| 1b. Thử `n ≤ num` | đồng dư | O(num) | O(1) | doocs 1 |
| 2. Thử `n ≤ 10` | chu kỳ mod 10 | O(1) | O(1) | **mặc định** |
| 3. Đệ quy có nhớ | trừ dần | O(num²/10) | O(num) | tổng quát |
| 4. Bảng tra | tiền tính | O(1) | O(1) | rất nhiều truy vấn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minimumNumbers(20, 5)` và `minimumNumbers(21, 5)`?

**Bài 2 (tính tay):** `num = 46`, `k = 8`. Tìm `n` nhỏ nhất và một tập chứng nhận.

**Bài 3 (điền TODO):** Viết `bt3_todo(num, k)` trả về **một tập** kích thước nhỏ nhất (danh sách), hoặc `None`.

**Bài 4 (tự viết lại):** Viết `bt4_todo(num, k)` cài cách **thử n = 1 … 10** không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(num, k)` trả về **mọi** kích thước `n` khả thi (danh sách tăng dần), O(num / 10 + 10).

In [32]:
def bt3_todo(num, k):
    # TODO: một tập nhỏ nhất
    pass

def bt4_todo(num, k):
    # TODO: thử n = 1..10
    pass

def bt5_todo(num, k):
    # TODO: mọi kích thước khả thi
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `5n mod 10 ∈ {0, 5}`. Mức 2: `20 ≡ 0` ⇒ `n = 2` (`10 ≤ 20`). Mức 3: `21 ≡ 1` không đạt ⇒ `−1`.

**Bài 2.** Mức 1: `8n mod 10`: `8, 6, 4, 2, 0, …`. Mức 2: `46 ≡ 6` ⇒ `n = 2` (`16 ≤ 46`). Mức 3: tập `[8, 38]`.

**Bài 3.** Mức 1: tính `n`. Mức 2: `k ≥ 1`: `[k]*(n−1) + [num − (n−1)k]`. Mức 3: `k = 0`: `[num]`; `num = 0`: `[]`.

**Bài 4.** Mức 1: `num == 0 ⇒ 0`. Mức 2: thử `n` trong `range(1, 11)`. Mức 3: điều kiện `n*k <= num and (num - n*k) % 10 == 0`.

**Bài 5.** Mức 1: `k ≥ 1`: `n₀` nhỏ nhất, rồi `n₀ + p·t` với `p = 10 / gcd(k, 10)` là **chu kỳ thật** của `n·k mod 10` (ví dụ `k = 5` ⇒ `p = 2`, không phải 10!), miễn `n·k ≤ num`. Mức 2: `k = 0`: mọi `n` với `10n ≤ num` (nếu `num ≡ 0`). Mức 3: `num = 0` ⇒ `[0]`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `2`, `−1`. **Bài 2:** `2`, tập `[8, 38]`.

Lưu ý khi làm bài 5: phiên bản đầu tiên của đáp án dùng bước nhảy 10 và bị assert bắt lỗi ngay (ví dụ `num = 20, k = 5` có các kích thước `2, 4` chứ không chỉ `2`) — chu kỳ của `n·k mod 10` là `10 / gcd(k, 10)`, nhỏ hơn 10 khi `k` chẵn hoặc chia hết cho 5.

In [33]:
assert Solution().minimumNumbers(20, 5) == 2 and Solution().minimumNumbers(21, 5) == -1
assert Solution().minimumNumbers(46, 8) == 2

def bt3(num, k):
    n = Solution().minimumNumbers(num, k)
    if n == -1: return None
    if n == 0: return []
    return [num] if k == 0 else [k] * (n - 1) + [num - (n - 1) * k]

def bt4(num, k):
    return enumerate_ten(num, k)

def bt5(num, k):
    if num == 0: return [0]
    if k == 0: return list(range(1, num // 10 + 1)) if num % 10 == 0 else []
    n0 = Solution().minimumNumbers(num, k)
    step = 10 // gcd(k, 10)                       # chu kỳ thật của n*k mod 10
    return [] if n0 == -1 else list(range(n0, num // k + 1, step))

def bt5_brute(num, k):
    coins = [c for c in range(1, num + 1) if c % 10 == k]
    reach = [set() for _ in range(num + 1)]; reach[0].add(0)
    for v in range(1, num + 1):
        for c in coins:
            if c > v: break
            reach[v] |= {x + 1 for x in reach[v - c]}
    return sorted(reach[num])

for k in range(10):
    for v in range(0, 120):
        s = bt3(v, k)
        assert (s is None) == (Solution().minimumNumbers(v, k) == -1)
        if s is not None: assert sum(s) == v and all(x > 0 and x % 10 == k for x in s)
        assert bt4(v, k) == brute(v, k)
        assert bt5(v, k) == bt5_brute(v, k), (v, k)
print(bt3(46, 8), bt5(100, 3), bt5(40, 0))
print("Đáp án bài tập: tất cả assert OK")

[8, 38] [10, 20, 30] [1, 2, 3, 4]
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
if num == 0: return 0                        # tập rỗng
for n in range(1, 11):                       # chu kỳ mod 10
    if n * k <= num and (num - n * k) % 10 == 0:
        return n                             # (n-1) số k + một số num-(n-1)k
return -1
```

**Checklist:**
- [ ] `num = 0` ⇒ 0 (kể cả `k = 0`).
- [ ] Thử tới `n = 10` (không phải 9).
- [ ] Kiểm tra `n·k ≤ num`.
- [ ] `k = 0`: số dương nhỏ nhất là 10, không dùng 0.